# Hybrid Retrieval and Reranking

This notebook is the missing bridge between **retrieval fundamentals** and full **RAG systems**.

We will build a small but realistic multi-stage retrieval stack and use it to answer five practical questions:

1. Why do **BM25** and **dense retrieval** fail on different queries?
2. How should we combine them with **score fusion**?
3. Why does a **reranker** improve top-of-list precision?
4. When does **query rewriting** help more than better embeddings?
5. How do we do **failure analysis** instead of trusting one average metric?

The goal is not benchmark performance. The goal is to build intuition for the retrieval stack that usually sits between a corpus and a generator.


## 1. Setup


In [ ]:
import math
import re
from collections import Counter
from dataclasses import dataclass
import importlib.util
from pathlib import Path

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import CSVLogger
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

repo_root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
utils_path = repo_root / 'src/aiml_notebooks/utils.py'
spec = importlib.util.spec_from_file_location('aiml_notebooks_utils', utils_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)
get_device = utils.get_device
set_seed = utils.set_seed

CONFIG = {
    'seed': 42,  # Random seed for reproducibility
    'max_length': 48,  # Maximum tokens kept from a query or document
    'dense_batch_size': 8,  # Batch size for the dense retriever
    'dense_embedding_dim': 48,  # Token embedding width for the dense retriever
    'dense_projection_dim': 32,  # Final retrieval embedding size
    'dense_learning_rate': 3e-3,  # Optimizer learning rate for dense training
    'dense_max_epochs': 18,  # Maximum dense retriever epochs
    'dense_patience': 4,  # Early stopping patience for dense training
    'temperature': 0.20,  # Contrastive temperature for dense retrieval
    'reranker_batch_size': 16,  # Batch size for reranker training
    'reranker_hidden_dim': 64,  # Hidden width for the reranker MLP head
    'reranker_learning_rate': 2e-3,  # Optimizer learning rate for reranker training
    'reranker_max_epochs': 18,  # Maximum reranker epochs
    'reranker_patience': 4,  # Early stopping patience for reranker training
    'fusion_alpha': 0.55,  # Weight on BM25 after score normalization
    'rrf_k': 60,  # Reciprocal rank fusion smoothing constant
    'first_stage_k': 5,  # Number of candidates kept before reranking
    'final_k': 3,  # Number of final results we care about most
}

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
print('Configuration loaded.')


### Random Seed and Device

We will still inspect the available device, but these models are intentionally tiny. To keep execution stable across machines, we will train on **CPU**.


In [ ]:
set_seed(CONFIG['seed'])
device = get_device(prefer_cpu=True)
trainer_accelerator = 'cpu'
print('Notebook training accelerator:', trainer_accelerator)


## 2. Build a Retrieval Playground

A bridge notebook needs a corpus with **controlled failure modes**.

The documents below mix lexical clues, paraphrases, acronyms, and operational language. That gives us queries where:

- BM25 should win because the wording matches exactly
- Dense retrieval should win because the query is phrased differently
- Query rewriting should win because the query is too short or acronym-heavy
- Reranking should win because several candidates are almost right


In [ ]:
@dataclass
class KnowledgeDoc:
    doc_id: str
    section: str
    title: str
    text: str
    train_queries: list[str]


@dataclass
class EvalQuery:
    query_id: str
    text: str
    relevant_docs: list[str]
    query_type: str
    needs_rewrite: bool = False


DOCUMENTS = [
    KnowledgeDoc(
        'doc_hybrid_search',
        'retrieval',
        'Hybrid retrieval mixes lexical and semantic signals',
        'Hybrid retrieval combines BM25 keyword matching with dense vector search. '
        'It is useful when some users type exact product terms and others use paraphrases. '
        'Reciprocal rank fusion is robust because it combines rank positions instead of raw score scales.',
        [
            'combine bm25 with dense retrieval',
            'merge keyword and vector search',
            'when should lexical and semantic ranking work together',
            'hybrid search with reciprocal rank fusion',
            'blend sparse and dense retrieval signals',
        ],
    ),
    KnowledgeDoc(
        'doc_reranking',
        'ranking',
        'Rerankers clean up the top of the list',
        'A reranker scores the query and candidate passage jointly after a fast first stage. '
        'Because it sees the pair together, it can demote near matches and promote the most specific evidence. '
        'Reranking usually improves precision at the very top of the list.',
        [
            'rerank top retrieved passages',
            'second stage relevance model',
            'improve precision at rank one',
            'score query document pairs jointly',
            'promote the most specific retrieved evidence',
        ],
    ),
    KnowledgeDoc(
        'doc_query_rewrite',
        'querying',
        'Query rewriting expands short or underspecified searches',
        'Query rewriting expands acronyms, restores omitted nouns, and injects domain synonyms before retrieval. '
        'It helps when users type shorthand such as ANN, SSO, or 429 without enough surrounding context. '
        'Rewriting is most helpful for short underspecified queries.',
        [
            'expand acronyms before search',
            'rewrite terse search queries',
            'add missing nouns to the query',
            'help short underspecified retrieval prompts',
            'improve retrieval for shorthand queries',
        ],
    ),
    KnowledgeDoc(
        'doc_chunking',
        'corpus',
        'Chunk size trades context against precision',
        'Small chunks isolate evidence but can lose surrounding context. '
        'Large chunks preserve context but introduce distracting terms that confuse ranking. '
        'Tune chunk size for the retriever, reranker, and generator together.',
        [
            'choose chunk size for retrieval',
            'tiny passages lose context',
            'large chunks add distracting terms',
            'chunking tradeoff for rag',
            'how passage length changes retrieval quality',
        ],
    ),
    KnowledgeDoc(
        'doc_ann',
        'indexing',
        'ANN indexes trade a little recall for speed',
        'Approximate nearest neighbor indexes reduce dense retrieval latency by scanning only a subset of vectors. '
        'Inverted files and graph indexes trade a little recall for much faster search. '
        'ANN only matters when exact dense search is too slow at your scale.',
        [
            'approximate nearest neighbor indexing',
            'speed up vector retrieval',
            'trade recall for retrieval latency',
            'ivf and graph index for embeddings',
            'faster approximate embedding lookup',
        ],
    ),
    KnowledgeDoc(
        'doc_filters',
        'indexing',
        'Metadata filters narrow the candidate set',
        'Metadata filters narrow the candidate set before scoring. '
        'Use them for tenant boundaries, language restrictions, document type, or freshness rules. '
        'Filters improve latency and precision when the query already implies a subset of the corpus.',
        [
            'metadata filters before ranking',
            'restrict retrieval by tenant',
            'freshness and language filters',
            'pre filter candidate set',
            'narrow search before similarity scoring',
        ],
    ),
    KnowledgeDoc(
        'doc_eval',
        'evaluation',
        'Evaluate recall early and ranking quality late',
        'Evaluate candidate generation with recall at k because missing the right passage early cannot be fixed later. '
        'Evaluate final ranking with MRR or nDCG because top order quality matters after reranking. '
        'Track failure slices by query type instead of only looking at macro averages.',
        [
            'recall at k for candidate generation',
            'evaluate reranking with ndcg',
            'mrr for final ranking',
            'slice retrieval failures by query type',
            'which retrieval metrics matter at each stage',
        ],
    ),
    KnowledgeDoc(
        'doc_rag_guard',
        'rag',
        'Bad retrieval causes unsupported RAG answers',
        'RAG quality depends on grounded context, not just generation quality. '
        'Retrieval failures show up as unsupported answers, wrong citations, or passages that mention the topic without containing the needed fact. '
        'Query level logging is essential for diagnosing these misses.',
        [
            'unsupported answers come from bad retrieval',
            'wrong citation due to wrong context',
            'rag grounding failures',
            'log query level retrieval misses',
            'retrieval errors behind hallucinated answers',
        ],
    ),
    KnowledgeDoc(
        'doc_auth',
        'operations',
        'Single sign-on issues often start at the identity provider',
        'Single sign-on troubleshooting starts with the identity provider and federation settings, not the embedding model. '
        'Expanding SSO to single sign-on in the query helps exact match retrievers find the right operations guide. '
        'Login loops often come from certificate or redirect mismatches.',
        [
            'single sign on troubleshooting',
            'identity provider login loop',
            'expand sso in the query',
            'auth guide for federated login',
            'certificate mismatch in single sign on flow',
        ],
    ),
    KnowledgeDoc(
        'doc_rate_limits',
        'operations',
        'Rate limits and quota language need normalization',
        'Rate limits and token quotas control throughput. '
        'Search for quota exhaustion, usage caps, or request throttling when users report HTTP 429 errors. '
        'Rewriting colloquial phrasing into platform terminology helps sparse retrievers.',
        [
            '429 quota exhaustion meaning',
            'usage cap and throttling',
            'rate limit troubleshooting',
            'rewrite colloquial quota errors',
            'what causes http 429 responses',
        ],
    ),
]

EVAL_QUERIES = [
    EvalQuery('q_keyword_embeddings', 'How do I combine keyword search with embeddings?', ['doc_hybrid_search'], 'semantic'),
    EvalQuery('q_second_stage', 'Why use a second-stage model after candidate retrieval?', ['doc_reranking'], 'semantic'),
    EvalQuery('q_acronym_help', 'How do I fix acronym-heavy search queries?', ['doc_query_rewrite'], 'lexical'),
    EvalQuery('q_tiny_chunks', 'When do small passages hurt answer quality?', ['doc_chunking'], 'semantic'),
    EvalQuery('q_ann_short', 'ann latency tradeoff', ['doc_ann', 'doc_query_rewrite'], 'acronym', True),
    EvalQuery('q_sso_short', 'sso login keeps looping', ['doc_auth', 'doc_query_rewrite'], 'acronym', True),
    EvalQuery('q_filters', 'How can search stay inside one tenant?', ['doc_filters'], 'lexical'),
    EvalQuery('q_candidate_metric', 'Which metric matters before reranking?', ['doc_eval'], 'semantic'),
    EvalQuery('q_unsupported_answers', 'Why does wrong evidence create unsupported RAG answers?', ['doc_rag_guard'], 'semantic'),
    EvalQuery('q_429', 'Why am I getting 429 usage cap errors?', ['doc_rate_limits', 'doc_query_rewrite'], 'acronym', True),
    EvalQuery('q_rrf', 'Why is reciprocal rank fusion safer than adding raw scores?', ['doc_hybrid_search'], 'lexical'),
    EvalQuery('q_specificity', 'How do I demote near matches and boost the most specific passage?', ['doc_reranking'], 'semantic'),
]

docs_df = pd.DataFrame([
    {
        'doc_id': doc.doc_id,
        'section': doc.section,
        'title': doc.title,
        'text': doc.text,
        'full_text': f'{doc.title}. {doc.text}',
    }
    for doc in DOCUMENTS
])

eval_queries_df = pd.DataFrame([
    {
        'query_id': query.query_id,
        'text': query.text,
        'relevant_docs': query.relevant_docs,
        'query_type': query.query_type,
        'needs_rewrite': query.needs_rewrite,
    }
    for query in EVAL_QUERIES
])

print(f'Documents: {len(docs_df)}')
print(f'Evaluation queries: {len(eval_queries_df)}')
display(docs_df[['doc_id', 'section', 'title']].head())
display(eval_queries_df[['query_id', 'query_type', 'needs_rewrite', 'text', 'relevant_docs']])


### Inspect the Failure Slices

Before ranking anything, it helps to know what kinds of queries we are testing. Acronym-heavy queries should behave differently from long semantic paraphrases.


In [ ]:
query_type_counts = eval_queries_df['query_type'].value_counts().rename_axis('query_type').reset_index(name='count')
display(query_type_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
docs_df['section'].value_counts().sort_values().plot(kind='barh', ax=axes[0], color='#4C78A8')
axes[0].set_title('Documents by section')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Section')

eval_queries_df.groupby(['query_type', 'needs_rewrite']).size().unstack(fill_value=0).plot(
    kind='bar',
    ax=axes[1],
    color=['#72B7B2', '#F58518'],
)
axes[1].set_title('Query types and rewrite demand')
axes[1].set_xlabel('Query type')
axes[1].set_ylabel('Count')
axes[1].legend(['No rewrite', 'Needs rewrite'])
plt.tight_layout()
plt.show()


### Why a Single Retriever Usually Fails

Some queries are long enough to express intent clearly. Others are just fragments like `ann latency tradeoff`. That means the retrieval stack needs more than one tool.


In [ ]:
def simple_token_count(text):
    return len(re.findall(r"[a-z0-9']+", text.lower()))

query_diagnostics = eval_queries_df[['query_id', 'query_type', 'needs_rewrite']].copy()
query_diagnostics['num_terms'] = eval_queries_df['text'].map(simple_token_count)
query_diagnostics['num_relevant_docs'] = eval_queries_df['relevant_docs'].map(len)
display(query_diagnostics)

fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(
    data=query_diagnostics,
    x='num_terms',
    y='num_relevant_docs',
    hue='query_type',
    style='needs_rewrite',
    s=130,
    ax=ax,
)
for _, row in query_diagnostics.iterrows():
    ax.text(row['num_terms'] + 0.05, row['num_relevant_docs'] + 0.02, row['query_id'], fontsize=9)
ax.set_title('Short ambiguous queries need extra help')
ax.set_xlabel('Query length (tokens)')
ax.set_ylabel('Number of relevant documents')
plt.tight_layout()
plt.show()


## 3. Tokenization and Encoding Helpers

We will use a lightweight lowercase tokenizer. The point of this notebook is the **retrieval stack**, not tokenizer engineering.


In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())


def build_vocabulary(texts):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = ['[PAD]', '[UNK]'] + sorted(counter)
    token_to_id = {token: idx for idx, token in enumerate(vocab)}
    return vocab, token_to_id


all_training_queries = [query for doc in DOCUMENTS for query in doc.train_queries]
vocab, token_to_id = build_vocabulary(list(docs_df['full_text']) + all_training_queries + list(eval_queries_df['text']))
pad_idx = token_to_id['[PAD]']
unk_idx = token_to_id['[UNK]']


def encode_text(text, max_length):
    token_ids = [token_to_id.get(token, unk_idx) for token in tokenize(text)[:max_length]]
    attention_mask = [1] * len(token_ids)
    if len(token_ids) < max_length:
        padding = [pad_idx] * (max_length - len(token_ids))
        token_ids += padding
        attention_mask += [0] * len(padding)
    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(attention_mask, dtype=torch.long)


sample_ids, sample_mask = encode_text(eval_queries_df.iloc[0]['text'], CONFIG['max_length'])
print('Vocabulary size:', len(vocab))
print('Encoded query shape:', tuple(sample_ids.shape), 'non-pad tokens:', int(sample_mask.sum()))


## 4. Prepare Supervision for a Tiny Dense Retriever

The previous notebook already taught dense retrieval from scratch. Here we only need a small semantic encoder so the hybrid system has a meaningful **dense signal**.


In [ ]:
dense_train_rows = []
dense_val_rows = []
for doc in DOCUMENTS:
    for query_text in doc.train_queries[:-1]:
        dense_train_rows.append({'query_text': query_text, 'doc_id': doc.doc_id})
    dense_val_rows.append({'query_text': doc.train_queries[-1], 'doc_id': doc.doc_id})

dense_train_df = pd.DataFrame(dense_train_rows)
dense_val_df = pd.DataFrame(dense_val_rows)

display(dense_train_df.head(10))
print('Dense training pairs:', len(dense_train_df))
print('Dense validation pairs:', len(dense_val_df))


### Wrap Query-Document Pairs in a Dataset

Each training example is one positive query-document pair. Other documents in the same batch will act as **in-batch negatives**.


In [ ]:
class DenseRetrievalDataset(Dataset):
    def __init__(self, query_frame, document_frame):
        doc_lookup = document_frame.set_index('doc_id')['full_text']
        self.samples = []
        for _, row in query_frame.iterrows():
            self.samples.append((row['query_text'], doc_lookup.loc[row['doc_id']]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        query_text, doc_text = self.samples[idx]
        query_ids, query_mask = encode_text(query_text, CONFIG['max_length'])
        doc_ids, doc_mask = encode_text(doc_text, CONFIG['max_length'])
        return query_ids, query_mask, doc_ids, doc_mask


dense_train_dataset = DenseRetrievalDataset(dense_train_df, docs_df)
dense_val_dataset = DenseRetrievalDataset(dense_val_df, docs_df)

dense_train_loader = DataLoader(dense_train_dataset, batch_size=CONFIG['dense_batch_size'], shuffle=True)
dense_val_loader = DataLoader(dense_val_dataset, batch_size=CONFIG['dense_batch_size'], shuffle=False)

print('Dense train batches:', len(dense_train_loader))
print('Dense validation batches:', len(dense_val_loader))


### Implement a Tiny Dual Encoder

This model is intentionally small. We only need a local semantic signal that captures paraphrases better than BM25.


In [ ]:
class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, projection_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.projection = nn.Linear(embedding_dim, projection_dim)

    def forward(self, input_ids, attention_mask):
        embeddings = self.embedding(input_ids)
        mask = attention_mask.unsqueeze(-1)
        pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        projected = self.projection(pooled)
        return F.normalize(projected, dim=-1)


class DualEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, projection_dim):
        super().__init__()
        self.encoder = TinyTextEncoder(vocab_size, embedding_dim, projection_dim)

    def encode_queries(self, input_ids, attention_mask):
        return self.encoder(input_ids, attention_mask)

    def encode_docs(self, input_ids, attention_mask):
        return self.encoder(input_ids, attention_mask)


class DualEncoderModule(L.LightningModule):
    def __init__(self, model, learning_rate, temperature):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.temperature = temperature

    def forward(self, query_ids, query_mask, doc_ids, doc_mask):
        query_embeddings = self.model.encode_queries(query_ids, query_mask)
        doc_embeddings = self.model.encode_docs(doc_ids, doc_mask)
        return query_embeddings, doc_embeddings

    def _shared_step(self, batch, stage):
        query_ids, query_mask, doc_ids, doc_mask = batch
        query_embeddings, doc_embeddings = self(query_ids, query_mask, doc_ids, doc_mask)
        logits = query_embeddings @ doc_embeddings.T / self.temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
        acc = (logits.argmax(dim=1) == labels).float().mean()
        self.log(f'{stage}_loss', loss, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        self.log(f'{stage}_acc', acc, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, 'val')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


### Train the Dense Retriever

We want the dense model to separate paraphrases from unrelated documents, not to become large or slow.


In [ ]:
dense_model = DualEncoder(len(vocab), CONFIG['dense_embedding_dim'], CONFIG['dense_projection_dim'])
dense_module = DualEncoderModule(dense_model, CONFIG['dense_learning_rate'], CONFIG['temperature'])
dense_logger = CSVLogger('logs', name='hybrid_retrieval_bridge', version='dense_retriever')
dense_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['dense_patience'],
    mode='min',
    verbose=False,
)

dense_trainer = L.Trainer(
    max_epochs=CONFIG['dense_max_epochs'],
    accelerator=trainer_accelerator,
    devices=1,
    logger=dense_logger,
    callbacks=[dense_early_stop],
    deterministic=True,
    enable_progress_bar=False,
    enable_checkpointing=False,
    num_sanity_val_steps=0,
    log_every_n_steps=1,
)

dense_trainer.fit(dense_module, dense_train_loader, dense_val_loader)
print('Dense retriever log dir:', dense_logger.log_dir)


### Plot the Dense Training Curves

The exact values do not matter much here. We mainly want to see that the semantic encoder learns a stable alignment signal.


In [ ]:
dense_metrics = pd.read_csv(f'{dense_logger.log_dir}/metrics.csv')
dense_train_metrics = dense_metrics[['epoch', 'train_loss', 'train_acc']].dropna().groupby('epoch').mean().reset_index()
dense_val_metrics = dense_metrics[['epoch', 'val_loss', 'val_acc']].dropna().groupby('epoch').mean().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(dense_train_metrics['epoch'], dense_train_metrics['train_loss'], marker='o', label='Train')
ax1.plot(dense_val_metrics['epoch'], dense_val_metrics['val_loss'], marker='s', label='Validation')
ax1.set_title('Dense retriever loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(dense_train_metrics['epoch'], dense_train_metrics['train_acc'], marker='o', label='Train')
ax2.plot(dense_val_metrics['epoch'], dense_val_metrics['val_acc'], marker='s', label='Validation')
ax2.set_title('Dense retriever in-batch accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()


## 5. Build Sparse and Dense First-Stage Rankers

Now we have two different retrieval signals:

- **Sparse**: exact lexical evidence with BM25
- **Dense**: semantic similarity from the dual encoder


In [ ]:
def encode_text_list(model, texts, encoder_name):
    encoder = model.encode_docs if encoder_name == 'doc' else model.encode_queries
    outputs = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            token_ids, attention_mask = encode_text(text, CONFIG['max_length'])
            embedding = encoder(token_ids.unsqueeze(0), attention_mask.unsqueeze(0))
            outputs.append(embedding.squeeze(0).cpu().numpy())
    return np.vstack(outputs)


dense_doc_embeddings = encode_text_list(dense_module.model, docs_df['full_text'].tolist(), 'doc')


def dense_rank(query_text):
    query_embedding = encode_text_list(dense_module.model, [query_text], 'query')[0]
    scores = dense_doc_embeddings @ query_embedding
    order = np.argsort(scores)[::-1]
    return order, scores


example_query = eval_queries_df.loc[eval_queries_df['query_id'] == 'q_second_stage', 'text'].item()
order, scores = dense_rank(example_query)
example_view = docs_df.iloc[order[:5]][['doc_id', 'title']].copy()
example_view['dense_score'] = scores[order[:5]]
print('Dense retrieval example:', example_query)
display(example_view)


### Implement BM25

BM25 remains the baseline because it is strong on exact wording, cheap to run, and easy to interpret.


In [ ]:
class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(text) for text in texts]
        self.doc_lengths = np.array([len(doc) for doc in self.docs], dtype=np.float32)
        self.avg_doc_length = float(np.mean(self.doc_lengths))
        self.term_doc_freq = Counter()
        self.term_freqs = []
        for doc in self.docs:
            freq = Counter(doc)
            self.term_freqs.append(freq)
            for term in freq:
                self.term_doc_freq[term] += 1
        self.num_docs = len(self.docs)

    def idf(self, term):
        df = self.term_doc_freq.get(term, 0)
        return math.log(1 + (self.num_docs - df + 0.5) / (df + 0.5))

    def score_all(self, query):
        scores = np.zeros(self.num_docs, dtype=np.float32)
        query_terms = tokenize(query)
        for doc_index, doc_freqs in enumerate(self.term_freqs):
            doc_length = self.doc_lengths[doc_index]
            score = 0.0
            for term in query_terms:
                tf = doc_freqs.get(term, 0)
                if tf == 0:
                    continue
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_length / self.avg_doc_length)
                score += self.idf(term) * numerator / denominator
            scores[doc_index] = score
        return scores

    def rank(self, query):
        scores = self.score_all(query)
        order = np.argsort(scores)[::-1]
        return order, scores


bm25 = BM25(docs_df['full_text'].tolist())
lexical_query = eval_queries_df.loc[eval_queries_df['query_id'] == 'q_rrf', 'text'].item()
order, scores = bm25.rank(lexical_query)
lexical_view = docs_df.iloc[order[:5]][['doc_id', 'title']].copy()
lexical_view['bm25_score'] = scores[order[:5]]
print('BM25 example:', lexical_query)
display(lexical_view)


### Compare Sparse and Dense on Hard Queries

A useful bridge notebook shows **why** we need the extra stage, not just that one metric goes up.


In [ ]:
def top_results_frame(query_text, method_name, order, scores, top_k=4):
    frame = docs_df.iloc[order[:top_k]][['doc_id', 'title']].copy()
    frame['score'] = scores[order[:top_k]]
    frame['method'] = method_name
    return frame[['method', 'doc_id', 'title', 'score']]


for query_id in ['q_keyword_embeddings', 'q_second_stage', 'q_ann_short']:
    query_row = eval_queries_df[eval_queries_df['query_id'] == query_id].iloc[0]
    bm25_order, bm25_scores = bm25.rank(query_row['text'])
    dense_order, dense_scores = dense_rank(query_row['text'])
    print(f"\nQuery: {query_row['text']}")
    print('Relevant docs:', query_row['relevant_docs'])
    display(pd.concat([
        top_results_frame(query_row['text'], 'BM25', bm25_order, bm25_scores),
        top_results_frame(query_row['text'], 'Dense', dense_order, dense_scores),
    ], ignore_index=True))


## 6. Score Fusion for Hybrid Retrieval

Hybrid retrieval works only if we combine the two rankers carefully.

Raw BM25 scores and dense cosine scores live on different scales. A direct sum is usually brittle, so we will use two fusion ideas:

- **Weighted fusion after normalization**
- **Reciprocal rank fusion (RRF)**


In [ ]:
def minmax_normalize(scores):
    scores = np.asarray(scores, dtype=np.float32)
    low = float(scores.min())
    high = float(scores.max())
    if high <= low:
        return np.zeros_like(scores)
    return (scores - low) / (high - low)


def weighted_hybrid_scores(query_text, alpha=CONFIG['fusion_alpha']):
    bm25_scores = bm25.score_all(query_text)
    _, dense_scores = dense_rank(query_text)
    combined = alpha * minmax_normalize(bm25_scores) + (1 - alpha) * minmax_normalize(dense_scores)
    return combined, bm25_scores, dense_scores


def reciprocal_rank_fusion_scores(query_text, k=CONFIG['rrf_k']):
    bm25_order, _ = bm25.rank(query_text)
    dense_order, _ = dense_rank(query_text)
    bm25_ranks = np.empty(len(docs_df), dtype=np.int32)
    dense_ranks = np.empty(len(docs_df), dtype=np.int32)
    bm25_ranks[bm25_order] = np.arange(1, len(docs_df) + 1)
    dense_ranks[dense_order] = np.arange(1, len(docs_df) + 1)
    return 1.0 / (k + bm25_ranks) + 1.0 / (k + dense_ranks)


def score_order(scores):
    return np.argsort(scores)[::-1]


def order_to_doc_ids(order):
    return docs_df.iloc[order]['doc_id'].tolist()


def weighted_hybrid_ranking_fn(query_text):
    scores, _, _ = weighted_hybrid_scores(query_text)
    return order_to_doc_ids(score_order(scores))


def rrf_hybrid_ranking_fn(query_text):
    return order_to_doc_ids(score_order(reciprocal_rank_fusion_scores(query_text)))


### Inspect Fusion on One Query

This table makes the normalization problem concrete. BM25 and dense scores are not comparable until we put them on a common footing.


In [ ]:
analysis_query = eval_queries_df.loc[eval_queries_df['query_id'] == 'q_keyword_embeddings', 'text'].item()
weighted_scores, bm25_raw, dense_raw = weighted_hybrid_scores(analysis_query)
rrf_scores = reciprocal_rank_fusion_scores(analysis_query)

fusion_view = docs_df[['doc_id', 'title']].copy()
fusion_view['bm25_raw'] = bm25_raw
fusion_view['dense_raw'] = dense_raw
fusion_view['bm25_norm'] = minmax_normalize(bm25_raw)
fusion_view['dense_norm'] = minmax_normalize(dense_raw)
fusion_view['weighted_hybrid'] = weighted_scores
fusion_view['rrf'] = rrf_scores
fusion_view = fusion_view.sort_values('weighted_hybrid', ascending=False)
display(fusion_view.head(6))

plot_frame = fusion_view.head(6).melt(id_vars=['doc_id'], value_vars=['bm25_norm', 'dense_norm', 'weighted_hybrid', 'rrf'])
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=plot_frame, x='doc_id', y='value', hue='variable', ax=ax)
ax.set_title('Normalized fusion signals for one query')
ax.set_xlabel('Document')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()


### Evaluate the Candidate Generators

At the first stage, the most important question is simple: **did the right document survive?**

That is why we care about **recall@k** before we care about reranking quality.


In [ ]:
def precision_at_k(relevant_docs, retrieved_docs, k):
    return len(set(relevant_docs) & set(retrieved_docs[:k])) / k


def recall_at_k(relevant_docs, retrieved_docs, k):
    return len(set(relevant_docs) & set(retrieved_docs[:k])) / len(set(relevant_docs))


def reciprocal_rank(relevant_docs, retrieved_docs):
    relevant_docs = set(relevant_docs)
    for rank, doc_id in enumerate(retrieved_docs, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank
    return 0.0


def dcg_at_k(relevant_docs, retrieved_docs, k):
    relevant_docs = set(relevant_docs)
    return sum((1.0 if doc_id in relevant_docs else 0.0) / math.log2(rank + 1)
               for rank, doc_id in enumerate(retrieved_docs[:k], start=1))


def ndcg_at_k(relevant_docs, retrieved_docs, k):
    ideal_dcg = dcg_at_k(relevant_docs, list(relevant_docs), min(k, len(relevant_docs)))
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(relevant_docs, retrieved_docs, k) / ideal_dcg


def bm25_ranking_fn(query_text):
    order, _ = bm25.rank(query_text)
    return order_to_doc_ids(order)


def dense_ranking_fn(query_text):
    order, _ = dense_rank(query_text)
    return order_to_doc_ids(order)


def evaluate_queries(query_frame, ranking_fn, top_k):
    rows = []
    for _, row in query_frame.iterrows():
        ranked_doc_ids = ranking_fn(row['text'])
        rows.append({
            'query_id': row['query_id'],
            'query_type': row['query_type'],
            'needs_rewrite': row['needs_rewrite'],
            'precision@k': precision_at_k(row['relevant_docs'], ranked_doc_ids, top_k),
            'recall@k': recall_at_k(row['relevant_docs'], ranked_doc_ids, top_k),
            'mrr': reciprocal_rank(row['relevant_docs'], ranked_doc_ids),
            'ndcg@k': ndcg_at_k(row['relevant_docs'], ranked_doc_ids, top_k),
            'top_docs': ranked_doc_ids[:top_k],
        })
    return pd.DataFrame(rows)


candidate_eval = {
    'BM25': evaluate_queries(eval_queries_df, bm25_ranking_fn, CONFIG['first_stage_k']),
    'Dense': evaluate_queries(eval_queries_df, dense_ranking_fn, CONFIG['first_stage_k']),
    'Weighted hybrid': evaluate_queries(eval_queries_df, weighted_hybrid_ranking_fn, CONFIG['first_stage_k']),
    'RRF hybrid': evaluate_queries(eval_queries_df, rrf_hybrid_ranking_fn, CONFIG['first_stage_k']),
}

candidate_summary = pd.DataFrame([
    {
        'method': name,
        'precision@5': frame['precision@k'].mean(),
        'recall@5': frame['recall@k'].mean(),
        'MRR': frame['mrr'].mean(),
        'nDCG@5': frame['ndcg@k'].mean(),
    }
    for name, frame in candidate_eval.items()
])

display(candidate_summary.style.format({'precision@5': '{:.2%}', 'recall@5': '{:.2%}', 'MRR': '{:.2f}', 'nDCG@5': '{:.2%}'}))

metric_plot = candidate_summary.melt(id_vars='method', var_name='metric', value_name='value')
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=metric_plot, x='metric', y='value', hue='method', ax=ax)
ax.set_title('Candidate generation summary')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 7. Build a Lightweight Reranker

Hybrid retrieval improves **candidate recall**, but it still leaves near-miss documents near the top.

A reranker is a second-stage model that looks at a **query-document pair together** and decides which candidate is the most specific fit.


In [ ]:
all_doc_ids = docs_df['doc_id'].tolist()


def build_reranker_pairs(query_frame, extra_easy_negative=True):
    rows = []
    for _, row in query_frame.iterrows():
        candidate_ids = rrf_hybrid_ranking_fn(row['query_text'])[:CONFIG['first_stage_k']]
        positive_doc_id = row['doc_id']
        if positive_doc_id not in candidate_ids:
            candidate_ids = [positive_doc_id] + [doc_id for doc_id in candidate_ids if doc_id != positive_doc_id]
            candidate_ids = candidate_ids[:CONFIG['first_stage_k']]
        if extra_easy_negative:
            for doc_id in all_doc_ids:
                if doc_id != positive_doc_id and doc_id not in candidate_ids:
                    candidate_ids = candidate_ids + [doc_id]
                    break
        for candidate_id in candidate_ids:
            rows.append({
                'query_text': row['query_text'],
                'doc_id': candidate_id,
                'label': float(candidate_id == positive_doc_id),
            })
    return pd.DataFrame(rows)


reranker_train_df = build_reranker_pairs(dense_train_df)
reranker_val_df = build_reranker_pairs(dense_val_df)

print('Reranker train pairs:', len(reranker_train_df))
print('Reranker validation pairs:', len(reranker_val_df))
display(reranker_train_df.head(12))


### Inspect the Pair Balance

A reranker does not scan the whole corpus. It only sees a small candidate set, so we want negatives that are **hard enough to be informative**.


In [ ]:
pair_balance = pd.DataFrame([
    {'split': 'train', 'pairs': len(reranker_train_df), 'positive_rate': reranker_train_df['label'].mean()},
    {'split': 'validation', 'pairs': len(reranker_val_df), 'positive_rate': reranker_val_df['label'].mean()},
])
display(pair_balance.style.format({'positive_rate': '{:.2%}'}))

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=reranker_train_df, x='label', ax=ax, palette=['#E45756', '#4C78A8'])
ax.set_title('Reranker training labels')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.set_xticklabels(['Negative', 'Positive'])
plt.tight_layout()
plt.show()


### Implement the Interaction Reranker

This is not a full transformer cross-encoder. Instead, it is a small interaction model that uses the query embedding, document embedding, and their pairwise interactions.

That keeps the notebook fast while preserving the main reranking intuition.


In [ ]:
class PairDataset(Dataset):
    def __init__(self, pair_frame, document_frame):
        doc_lookup = document_frame.set_index('doc_id')['full_text']
        self.samples = []
        for _, row in pair_frame.iterrows():
            self.samples.append((row['query_text'], doc_lookup.loc[row['doc_id']], row['label']))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        query_text, doc_text, label = self.samples[idx]
        query_ids, query_mask = encode_text(query_text, CONFIG['max_length'])
        doc_ids, doc_mask = encode_text(doc_text, CONFIG['max_length'])
        return query_ids, query_mask, doc_ids, doc_mask, torch.tensor(label, dtype=torch.float32)


class InteractionReranker(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.query_projection = nn.Linear(embedding_dim, hidden_dim)
        self.doc_projection = nn.Linear(embedding_dim, hidden_dim)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def encode(self, input_ids, attention_mask, projection):
        embeddings = self.embedding(input_ids)
        mask = attention_mask.unsqueeze(-1)
        pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return torch.tanh(projection(pooled))

    def forward(self, query_ids, query_mask, doc_ids, doc_mask):
        query_vec = self.encode(query_ids, query_mask, self.query_projection)
        doc_vec = self.encode(doc_ids, doc_mask, self.doc_projection)
        features = torch.cat([query_vec, doc_vec, torch.abs(query_vec - doc_vec), query_vec * doc_vec], dim=-1)
        return self.scorer(features).squeeze(-1)


class RerankerModule(L.LightningModule):
    def __init__(self, model, learning_rate):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.loss_fn = nn.BCEWithLogitsLoss()

    def _shared_step(self, batch, stage):
        query_ids, query_mask, doc_ids, doc_mask, labels = batch
        logits = self.model(query_ids, query_mask, doc_ids, doc_mask)
        loss = self.loss_fn(logits, labels)
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        acc = (predictions == labels).float().mean()
        self.log(f'{stage}_loss', loss, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        self.log(f'{stage}_acc', acc, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, 'val')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


reranker_train_loader = DataLoader(PairDataset(reranker_train_df, docs_df), batch_size=CONFIG['reranker_batch_size'], shuffle=True)
reranker_val_loader = DataLoader(PairDataset(reranker_val_df, docs_df), batch_size=CONFIG['reranker_batch_size'], shuffle=False)

print('Reranker train batches:', len(reranker_train_loader))
print('Reranker validation batches:', len(reranker_val_loader))


### Train the Reranker

The reranker only learns inside a small candidate set. That is why its job is **precision**, not global recall.


In [ ]:
reranker_model = InteractionReranker(len(vocab), CONFIG['dense_embedding_dim'], CONFIG['reranker_hidden_dim'])
reranker_module = RerankerModule(reranker_model, CONFIG['reranker_learning_rate'])
reranker_logger = CSVLogger('logs', name='hybrid_retrieval_bridge', version='reranker')
reranker_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['reranker_patience'],
    mode='min',
    verbose=False,
)

reranker_trainer = L.Trainer(
    max_epochs=CONFIG['reranker_max_epochs'],
    accelerator=trainer_accelerator,
    devices=1,
    logger=reranker_logger,
    callbacks=[reranker_early_stop],
    deterministic=True,
    enable_progress_bar=False,
    enable_checkpointing=False,
    num_sanity_val_steps=0,
    log_every_n_steps=1,
)

reranker_trainer.fit(reranker_module, reranker_train_loader, reranker_val_loader)
print('Reranker log dir:', reranker_logger.log_dir)


### Plot the Reranker Training Curves

A useful reranker should quickly separate positives from hard negatives inside the candidate pool.


In [ ]:
reranker_metrics = pd.read_csv(f'{reranker_logger.log_dir}/metrics.csv')
reranker_train_metrics = reranker_metrics[['epoch', 'train_loss', 'train_acc']].dropna().groupby('epoch').mean().reset_index()
reranker_val_metrics = reranker_metrics[['epoch', 'val_loss', 'val_acc']].dropna().groupby('epoch').mean().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(reranker_train_metrics['epoch'], reranker_train_metrics['train_loss'], marker='o', label='Train')
ax1.plot(reranker_val_metrics['epoch'], reranker_val_metrics['val_loss'], marker='s', label='Validation')
ax1.set_title('Reranker loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(reranker_train_metrics['epoch'], reranker_train_metrics['train_acc'], marker='o', label='Train')
ax2.plot(reranker_val_metrics['epoch'], reranker_val_metrics['val_acc'], marker='s', label='Validation')
ax2.set_title('Reranker accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()


### Apply the Reranker to Hybrid Candidates

The first stage will keep only a few candidates. The reranker then reorders that short list and leaves the rest of the hybrid ranking behind it.


In [ ]:
doc_lookup = docs_df.set_index('doc_id')


def reranker_score(query_text, doc_text):
    reranker_module.model.eval()
    with torch.no_grad():
        query_ids, query_mask = encode_text(query_text, CONFIG['max_length'])
        doc_ids, doc_mask = encode_text(doc_text, CONFIG['max_length'])
        logit = reranker_module.model(
            query_ids.unsqueeze(0),
            query_mask.unsqueeze(0),
            doc_ids.unsqueeze(0),
            doc_mask.unsqueeze(0),
        )
    return float(torch.sigmoid(logit).item())


def reranked_doc_ids(query_text, candidate_k=CONFIG['first_stage_k']):
    base_order = rrf_hybrid_ranking_fn(query_text)
    candidate_ids = base_order[:candidate_k]
    scored_candidates = []
    for doc_id in candidate_ids:
        doc_text = doc_lookup.loc[doc_id, 'full_text']
        scored_candidates.append((doc_id, reranker_score(query_text, doc_text)))
    reranked_candidates = [doc_id for doc_id, _ in sorted(scored_candidates, key=lambda item: item[1], reverse=True)]
    remaining = [doc_id for doc_id in base_order if doc_id not in candidate_ids]
    return reranked_candidates + remaining


def compare_pipeline_for_query(query_id):
    query_row = eval_queries_df[eval_queries_df['query_id'] == query_id].iloc[0]
    hybrid_top = rrf_hybrid_ranking_fn(query_row['text'])[:CONFIG['first_stage_k']]
    reranked_top = reranked_doc_ids(query_row['text'])[:CONFIG['first_stage_k']]
    comparison = pd.DataFrame({
        'hybrid_top': hybrid_top,
        'reranked_top': reranked_top,
    })
    display(comparison)


for query_id in ['q_second_stage', 'q_specificity', 'q_keyword_embeddings']:
    query_text = eval_queries_df.loc[eval_queries_df['query_id'] == query_id, 'text'].item()
    print(f'Query: {query_text}')
    compare_pipeline_for_query(query_id)


### Evaluate Final Ranking Quality

Now we switch from the candidate-generation question to the final ranking question:

- Did the relevant document survive? That was **recall@k**.
- Is the best evidence near the top? That is where **MRR** and **nDCG** matter.


In [ ]:
def rerank_ranking_fn(query_text):
    return reranked_doc_ids(query_text)


final_eval = {
    'BM25': evaluate_queries(eval_queries_df, bm25_ranking_fn, CONFIG['final_k']),
    'Dense': evaluate_queries(eval_queries_df, dense_ranking_fn, CONFIG['final_k']),
    'Weighted hybrid': evaluate_queries(eval_queries_df, weighted_hybrid_ranking_fn, CONFIG['final_k']),
    'RRF hybrid': evaluate_queries(eval_queries_df, rrf_hybrid_ranking_fn, CONFIG['final_k']),
    'Hybrid + rerank': evaluate_queries(eval_queries_df, rerank_ranking_fn, CONFIG['final_k']),
}

final_summary = pd.DataFrame([
    {
        'method': name,
        'precision@3': frame['precision@k'].mean(),
        'recall@3': frame['recall@k'].mean(),
        'MRR': frame['mrr'].mean(),
        'nDCG@3': frame['ndcg@k'].mean(),
    }
    for name, frame in final_eval.items()
])

display(final_summary.style.format({'precision@3': '{:.2%}', 'recall@3': '{:.2%}', 'MRR': '{:.2f}', 'nDCG@3': '{:.2%}'}))

relevant_lookup = eval_queries_df.set_index('query_id')['relevant_docs'].to_dict()
hit_frame = pd.DataFrame({'query_id': eval_queries_df['query_id']})
for method_name, frame in final_eval.items():
    hit_frame[method_name] = [
        int(top_docs[0] in set(relevant_lookup[query_id]))
        for query_id, top_docs in zip(frame['query_id'], frame['top_docs'])
    ]

heatmap_frame = hit_frame.set_index('query_id')
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(heatmap_frame, annot=True, cmap='Blues', cbar=False, ax=ax)
ax.set_title('Did each method get the top result right?')
ax.set_xlabel('Method')
ax.set_ylabel('Query')
plt.tight_layout()
plt.show()


## 8. Query Rewriting

Some misses are not about ranking at all. They come from the query being too short or too colloquial.

A practical system often fixes that upstream by expanding acronyms or adding domain terms before retrieval starts.


In [ ]:
PHRASE_REWRITES = [
    (r'\bann\b', 'approximate nearest neighbor'),
    (r'\bsso\b', 'single sign on identity provider'),
    (r'\b429\b', 'http 429 rate limit quota exhaustion'),
]

TOKEN_EXPANSIONS = {
    'keyword': 'keyword lexical',
    'embeddings': 'embeddings dense vectors',
}


def rewrite_query(query_text):
    rewritten = query_text.lower()
    for pattern, replacement in PHRASE_REWRITES:
        rewritten = re.sub(pattern, replacement, rewritten)
    for token, expansion in TOKEN_EXPANSIONS.items():
        rewritten = re.sub(rf'\b{re.escape(token)}\b', expansion, rewritten)
    return rewritten


rewrite_examples = eval_queries_df[eval_queries_df['needs_rewrite']][['query_id', 'text']].copy()
rewrite_examples['rewritten'] = rewrite_examples['text'].map(rewrite_query)
display(rewrite_examples)


def rewrite_rerank_ranking_fn(query_text):
    return reranked_doc_ids(rewrite_query(query_text))


rewrite_overall_eval = evaluate_queries(eval_queries_df, rewrite_rerank_ranking_fn, CONFIG['final_k'])
rewrite_slice_eval = evaluate_queries(eval_queries_df[eval_queries_df['needs_rewrite']], rewrite_rerank_ranking_fn, CONFIG['final_k'])
base_slice_eval = evaluate_queries(eval_queries_df[eval_queries_df['needs_rewrite']], rerank_ranking_fn, CONFIG['final_k'])

rewrite_summary = pd.DataFrame([
    {
        'slice': 'All queries',
        'method': 'Hybrid + rerank',
        'MRR': final_eval['Hybrid + rerank']['mrr'].mean(),
        'nDCG@3': final_eval['Hybrid + rerank']['ndcg@k'].mean(),
    },
    {
        'slice': 'All queries',
        'method': 'Rewrite + hybrid + rerank',
        'MRR': rewrite_overall_eval['mrr'].mean(),
        'nDCG@3': rewrite_overall_eval['ndcg@k'].mean(),
    },
    {
        'slice': 'Rewrite-needed only',
        'method': 'Hybrid + rerank',
        'MRR': base_slice_eval['mrr'].mean(),
        'nDCG@3': base_slice_eval['ndcg@k'].mean(),
    },
    {
        'slice': 'Rewrite-needed only',
        'method': 'Rewrite + hybrid + rerank',
        'MRR': rewrite_slice_eval['mrr'].mean(),
        'nDCG@3': rewrite_slice_eval['ndcg@k'].mean(),
    },
])

display(rewrite_summary.style.format({'MRR': '{:.2f}', 'nDCG@3': '{:.2%}'}))


## 9. Failure Analysis

Averages hide structure. Failure analysis asks **which class of query still breaks the system** after every improvement.


In [ ]:
method_rankings = {
    'BM25': bm25_ranking_fn,
    'Dense': dense_ranking_fn,
    'RRF hybrid': rrf_hybrid_ranking_fn,
    'Hybrid + rerank': rerank_ranking_fn,
    'Rewrite + hybrid + rerank': rewrite_rerank_ranking_fn,
}

failure_rows = []
for _, row in eval_queries_df.iterrows():
    relevant = set(row['relevant_docs'])
    ranked_lists = {name: ranking_fn(row['text']) for name, ranking_fn in method_rankings.items()}
    failure_rows.append({
        'query_id': row['query_id'],
        'query_type': row['query_type'],
        'needs_rewrite': row['needs_rewrite'],
        'bm25_top1': ranked_lists['BM25'][0],
        'dense_top1': ranked_lists['Dense'][0],
        'rrf_top1': ranked_lists['RRF hybrid'][0],
        'rerank_top1': ranked_lists['Hybrid + rerank'][0],
        'rewrite_rerank_top1': ranked_lists['Rewrite + hybrid + rerank'][0],
        'bm25_hit@1': int(ranked_lists['BM25'][0] in relevant),
        'dense_hit@1': int(ranked_lists['Dense'][0] in relevant),
        'rrf_hit@1': int(ranked_lists['RRF hybrid'][0] in relevant),
        'rerank_hit@1': int(ranked_lists['Hybrid + rerank'][0] in relevant),
        'rewrite_rerank_hit@1': int(ranked_lists['Rewrite + hybrid + rerank'][0] in relevant),
    })

failure_table = pd.DataFrame(failure_rows)
display(failure_table)

slice_summary = failure_table.groupby('query_type')[['bm25_hit@1', 'dense_hit@1', 'rrf_hit@1', 'rerank_hit@1', 'rewrite_rerank_hit@1']].mean()
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(slice_summary, annot=True, cmap='Blues', vmin=0.0, vmax=1.0, ax=ax)
ax.set_title('Hit@1 by query slice')
ax.set_xlabel('Method')
ax.set_ylabel('Query type')
plt.tight_layout()
plt.show()

decision_matrix = pd.DataFrame([
    ['BM25', 'Exact terminology, product names, transparent scoring', 'Misses paraphrases and shorthand'],
    ['Dense retrieval', 'Semantic overlap and rephrased intent', 'Can rank broad topical matches too highly'],
    ['Hybrid fusion', 'Robust first-stage candidate generation', 'Still leaves near-miss ordering problems'],
    ['Reranker', 'Top-of-list precision and specificity', 'Cannot recover a document that never survived stage one'],
    ['Query rewriting', 'Acronyms, shorthand, colloquial phrasing', 'Bad rewrites can inject the wrong intent'],
], columns=['Technique', 'When it helps most', 'Typical failure mode'])
display(decision_matrix)


## 10. Put the Whole Stack Together

The practical retrieval recipe is now straightforward:

1. Rewrite underspecified queries when needed.
2. Retrieve candidates with a hybrid first stage.
3. Rerank only the small candidate pool.
4. Send the best passages to the next stage, such as an answer generator.


In [ ]:
def retrieve(query_text, apply_rewrite=True, first_stage_k=CONFIG['first_stage_k'], final_k=CONFIG['final_k']):
    effective_query = rewrite_query(query_text) if apply_rewrite else query_text
    candidate_ids = rrf_hybrid_ranking_fn(effective_query)[:first_stage_k]
    final_ids = reranked_doc_ids(effective_query, candidate_k=first_stage_k)[:final_k]
    candidate_frame = docs_df.set_index('doc_id').loc[candidate_ids, ['title', 'section']].reset_index()
    final_frame = docs_df.set_index('doc_id').loc[final_ids, ['title', 'section']].reset_index()
    return effective_query, candidate_frame, final_frame


for query_text in [
    'ann latency tradeoff',
    'How do I combine keyword search with embeddings?',
    'sso login keeps looping',
]:
    effective_query, candidate_frame, final_frame = retrieve(query_text)
    print(f'Original query: {query_text}')
    print(f'Effective query: {effective_query}')
    print('First-stage candidates:')
    display(candidate_frame)
    print('Final reranked results:')
    display(final_frame)


## 11. Key Takeaways

This notebook should leave you with one mental model:

**retrieval is a pipeline, not a single score.**

That is the conceptual handoff you need before building a full RAG system.


In [ ]:
print('Key takeaways:')
print('1. Sparse and dense retrieval fail on different queries, so hybrid systems are usually safer than either one alone.')
print('2. Score fusion needs normalization or rank-based combination because raw BM25 and dense scores are not directly comparable.')
print('3. Candidate generation is about recall, while reranking is about precision at the top of the list.')
print('4. Query rewriting fixes a different class of problem: missing or underspecified user language.')
print('5. Failure analysis by slice is more informative than one overall metric average.')
print('6. This is the retrieval stack that feeds a full RAG pipeline in the next notebook.')
